# Post-training — SFT, DPO, GRPO

Pre-trained LLMs predict the next token but don't follow instructions or have values.
Post-training aligns them to be helpful assistants.

**Pipeline:**
```
Pre-trained model (raw next-token predictor)
    ↓ SFT (Supervised Fine-Tuning)
Instruction-following model
    ↓ DPO / RLHF / GRPO
Aligned assistant (helpful, harmless, honest)
```

**Install:** `pip install trl transformers peft datasets`

## 1. SFT — Supervised Fine-Tuning

Fine-tune on (instruction, response) pairs using standard cross-entropy loss.
Teaches the model the format and style of responses.

**Data format:**
```json
{"prompt": "What is the capital of France?", "completion": "The capital of France is Paris."}
```

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import torch

# --- Synthetic dataset ---
data = [
    {"text": "<|user|>What is PyTorch?<|assistant|>PyTorch is an open-source deep learning framework."},
    {"text": "<|user|>What is a tensor?<|assistant|>A tensor is an n-dimensional array used in ML."},
    {"text": "<|user|>What is fine-tuning?<|assistant|>Fine-tuning adapts a pre-trained model to a specific task."},
]
dataset = Dataset.from_list(data)

# --- Model ---
model_name = 'gpt2'
model      = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer  = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token

# --- Trainer ---
training_args = SFTConfig(
    output_dir='./sft_output',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    logging_steps=1,
    save_strategy='no',
    dataset_text_field='text',
    max_seq_length=128,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()
print('SFT training complete')

## 2. DPO — Direct Preference Optimization

Teaches the model to prefer "chosen" responses over "rejected" ones.
No reward model needed — directly optimizes the preference objective.

**Data format:**
```json
{
  "prompt":   "How do I sort a list in Python?",
  "chosen":   "Use list.sort() or sorted(list).",
  "rejected": "Just try things until it works."
}
```

In [ ]:
from trl import DPOConfig, DPOTrainer

# --- Synthetic preference dataset ---
pref_data = [
    {
        "prompt":   "What is 2+2?",
        "chosen":   "2+2 equals 4.",
        "rejected": "I don't know, math is hard."
    },
    {
        "prompt":   "Write a Python hello world",
        "chosen":   'print(\"Hello, World!\")',
        "rejected": "I cannot write code."
    },
]
pref_dataset = Dataset.from_list(pref_data)

# Reload fresh model
model     = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

dpo_args = DPOConfig(
    output_dir='./dpo_output',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=5e-5,
    beta=0.1,           # KL penalty — how much to stay close to reference model
    logging_steps=1,
    save_strategy='no',
    max_length=128,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,    # None = auto-create reference model from a copy
    args=dpo_args,
    train_dataset=pref_dataset,
    tokenizer=tokenizer,
)

dpo_trainer.train()
print('DPO training complete')

## 3. GRPO — Group Relative Policy Optimization

Used in DeepSeek-R1 and reasoning models. Instead of a reference model, it:
1. Generates multiple completions per prompt
2. Scores each with a reward function
3. Uses the group's relative scores as the baseline

Great for tasks with verifiable rewards (math, code correctness).

```
Prompt: "What is 15 × 7?"
Gen 1: "105"     → reward 1.0 (correct)
Gen 2: "95"      → reward 0.0 (wrong)
Gen 3: "105"     → reward 1.0 (correct)
Gen 4: "100"     → reward 0.0 (wrong)
Baseline = mean reward = 0.5
Update: reinforce gen 1 and 3 (above baseline), penalize gen 2 and 4
```

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# --- Simple math reward function ---
def reward_fn(completions, prompts=None, **kwargs):
    """Reward 1.0 if completion contains a number, else 0.0"""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0].get('content', '')
        has_number = any(c.isdigit() for c in text)
        rewards.append(1.0 if has_number else 0.0)
    return rewards


math_data = [
    {"prompt": "What is 5 + 3?"},
    {"prompt": "What is 10 - 4?"},
    {"prompt": "What is 3 × 7?"},
]
math_dataset = Dataset.from_list(math_data)

model     = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

grpo_args = GRPOConfig(
    output_dir='./grpo_output',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=1e-5,
    num_generations=4,   # generate 4 completions per prompt to form the group
    max_new_tokens=20,
    logging_steps=1,
    save_strategy='no',
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=grpo_args,
    train_dataset=math_dataset,
    tokenizer=tokenizer,
)

grpo_trainer.train()
print('GRPO training complete')

## When to use which

| Method | Data needed | Use case |
|---|---|---|
| SFT | (prompt, response) pairs | Teach format, style, basic instruction following |
| DPO | (prompt, chosen, rejected) | Improve quality, reduce bad outputs, align values |
| GRPO | Prompts + reward function | Reasoning (math, code), tasks with verifiable answers |

**Typical pipeline:** SFT first → then DPO or GRPO on top of the SFT model.